# Topic 6 — Cloud SQL: Structured Ticket History

Requires: the Cloud SQL commands in `../commands.md` already run (same instance as the original topic). Uses `support_customers`/`support_tickets` — different table names than the original module's `memories` table.

In [1]:
import sqlalchemy
from google.cloud.sql.connector import Connector
from setup import PROJECT_ID, REGION, CLOUD_SQL_INSTANCE_NAME, CLOUD_SQL_DB_NAME, CLOUD_SQL_USER, CLOUD_SQL_PASSWORD

connector = Connector()

def getconn():
    return connector.connect(
        f"{PROJECT_ID}:{REGION}:{CLOUD_SQL_INSTANCE_NAME}",
        "pg8000",
        user=CLOUD_SQL_USER,
        password=CLOUD_SQL_PASSWORD,
        db=CLOUD_SQL_DB_NAME,
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)

### Two related tables

In [2]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS support_customers (
            id SERIAL PRIMARY KEY, name TEXT, plan_tier TEXT
        )
    """))
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS support_tickets (
            id SERIAL PRIMARY KEY,
            customer_id INT REFERENCES support_customers(id),
            opened_at TIMESTAMP,
            resolved_at TIMESTAMP
        )
    """))
    conn.commit()
print("Tables ready.")

Tables ready.


### Sample data

In [3]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text("DELETE FROM support_tickets"))
    conn.execute(sqlalchemy.text("DELETE FROM support_customers"))

    conn.execute(sqlalchemy.text(
        "INSERT INTO support_customers (name, plan_tier) VALUES (:name, :tier)"
    ), [{"name": "Acme Corp", "tier": "Enterprise"}, {"name": "Small Biz Co", "tier": "Starter"}])
    conn.commit()

    customers = conn.execute(sqlalchemy.text("SELECT id, name FROM support_customers")).fetchall()

    conn.execute(sqlalchemy.text("""
        INSERT INTO support_tickets (customer_id, opened_at, resolved_at)
        VALUES (:cid, :opened, :resolved)
    """), [
        {"cid": customers[0].id, "opened": "2026-01-01 09:00", "resolved": "2026-01-01 11:00"},
        {"cid": customers[0].id, "opened": "2026-01-05 10:00", "resolved": "2026-01-05 22:00"},
        {"cid": customers[1].id, "opened": "2026-01-03 14:00", "resolved": "2026-01-03 15:00"},
    ])
    conn.commit()
print("Sample tickets inserted.")

Sample tickets inserted.


### The payoff — average resolution time per customer

In [4]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text("""
        SELECT c.name, AVG(EXTRACT(EPOCH FROM (t.resolved_at - t.opened_at)) / 3600) AS avg_hours
        FROM support_customers c JOIN support_tickets t ON c.id = t.customer_id
        GROUP BY c.name
    """))
    for row in result:
        print(f"{row.name}: {row.avg_hours:.1f} hours average resolution time")

Small Biz Co: 1.0 hours average resolution time
Acme Corp: 7.0 hours average resolution time


In [5]:
connector.close()